In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
insurance_data = pd.read_csv("../data/insurance.csv")

x = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]

x = pd.get_dummies(x, columns=["region"], drop_first=True, dtype=int)

x["sex"] = x["sex"].map({"male": 1, "female": 0})
x["smoker"] = x["smoker"].map({"yes": 1, "no": 0})

x["age_smoker"] = x["age"] * x["smoker"]
x["bmi_smoker"] = x["bmi"] * x["smoker"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [ ]:
alphas = [0.1, 0.5, 1.0, 5.0, 10.0]
mse_values = []

for alpha in alphas:
    lasso_model = Lasso(alpha=alpha)
    lasso_model.fit(x_train, y_train)

    y_pred = lasso_model.predict(x_test)

    mse = mean_squared_error(y_test, y_pred)
    mse_values.append(mse)

    print(f"Alpha: {alpha}, Mean Squared Error: {mse}")

lowest_mse = min(mse_values)
best_alpha = alphas[mse_values.index(lowest_mse)]

print(f"\nBest Alpha: {best_alpha}")
print(f"Lowest MSE: {lowest_mse}")

# Visualize the relationship between alpha and MSE.
sns.lineplot(x=alphas, y=mse_values, marker="o")

In [ ]:
# Lasso regression with the best alpha using LassoCV, which uses
# cross-validation to find the best alpha.
from sklearn.linear_model import LassoCV

alphas = [0.1, 0.5, 1.0, 5.0, 10.0]

lasso_cv_model = LassoCV(alphas=alphas, cv=5, max_iter=1000, random_state=42)

lasso_cv_model.fit(x_train, y_train)

print(f"Best Alpha from LassoCV: {lasso_cv_model.alpha_}")

y_pred_cv = lasso_cv_model.predict(x_test)
mse = mean_squared_error(y_test, y_pred_cv)
print(f"Mean Squared Error with LassoCV: {mse}")
r2 = r2_score(y_test, y_pred_cv)
print(f"R-squared with LassoCV: {r2}")